# Tích hợp dữ liệu phục vụ dự báo CPI giao thông

Notebook này thực hiện tích hợp các bộ dữ liệu đã được tiền xử lý về cùng tần suất tháng, bao gồm CPI nhóm Giao thông, giá xăng RON95, giá dầu Diesel, giá dầu thô Brent, WTI và tỷ giá USD/VND.

In [3]:
import pandas as pd

In [4]:
cpi = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/cpi_transport_monthly.csv")

fuel = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/fuel_prices_monthly.csv")

brent = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/brent_monthly.csv")

wti = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/wti_monthly.csv")

usd_vnd = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/usd_vnd_monthly.csv")

print("CPI:", cpi.columns.tolist())
print("Fuel:", fuel.columns.tolist())
print("Brent:", brent.columns.tolist())
print("WTI:", wti.columns.tolist())
print("USD/VND:", usd_vnd.columns.tolist())

CPI: ['MonthYear', 'CPI']
Fuel: ['MonthYear', 'Diesel', 'RON95']
Brent: ['MonthYear', 'Brent']
WTI: ['MonthYear', 'WTI']
USD/VND: ['MonthYear', 'USD_VND']


In [5]:
cpi["MonthYear"] = pd.to_datetime(cpi["MonthYear"]).dt.to_period("M")
fuel["MonthYear"] = pd.to_datetime(fuel["MonthYear"]).dt.to_period("M")
brent["MonthYear"] = pd.to_datetime(brent["MonthYear"]).dt.to_period("M")
wti["MonthYear"] = pd.to_datetime(wti["MonthYear"]).dt.to_period("M")
usd_vnd["MonthYear"] = pd.to_datetime(usd_vnd["MonthYear"]).dt.to_period("M")

print(cpi["MonthYear"].dtype)
print(fuel["MonthYear"].dtype)
print(brent["MonthYear"].dtype)
print(wti["MonthYear"].dtype)
print(usd_vnd["MonthYear"].dtype)

period[M]
period[M]
period[M]
period[M]
period[M]


In [6]:
dataset = (
    cpi
    .merge(fuel, on="MonthYear", how="left")
    .merge(brent, on="MonthYear", how="left")
    .merge(wti, on="MonthYear", how="left")
    .merge(usd_vnd, on="MonthYear", how="left")
)

dataset.head()

,MonthYear,CPI,Diesel,RON95,Brent,WTI,USD_VND
0,2012-01,0.66,21100.00,21800.00,110.69,100.27,20828.0
1,2012-02,0.23,21100.00,21800.00,119.33,102.20,20828.0
2,2012-03,1.08,21341.94,23090.32,125.45,106.16,20828.0
3,2012-04,2.67,21400.00,23400.00,119.75,103.32,20828.0
4,2012-05,1.32,21432.26,23522.58,110.34,94.66,20828.0


In [7]:
dataset.shape

(156, 7)

In [8]:
dataset.isna().sum()

MonthYear    0
CPI          0
Diesel       0
RON95        0
Brent        0
WTI          0
USD_VND      0
dtype: int64

## Bổ sung biến giả Tết Nguyên đán

Biến `Dummy_Tet` được sử dụng để phản ánh ảnh hưởng của dịp Tết Nguyên đán đến nhu cầu đi lại và CPI nhóm Giao thông.

- `Dummy_Tet = 1`: tháng chịu tác động chính của Tết Nguyên đán.
- `Dummy_Tet = 0`: các tháng còn lại.

In [9]:
tet_months = {
    2012: 1,
    2013: 2,
    2014: 1,
    2015: 2,
    2016: 2,
    2017: 1,
    2018: 2,
    2019: 2,
    2020: 1,
    2021: 2,
    2022: 2,
    2023: 1,
    2024: 2
}

dataset["Dummy_Tet"] = dataset["MonthYear"].apply(
    lambda x: 1 if tet_months.get(x.year) == x.month else 0
)

dataset.loc[
    dataset["Dummy_Tet"] == 1,
    ["MonthYear", "Dummy_Tet"]
]

,MonthYear,Dummy_Tet
0,2012-01,1
13,2013-02,1
24,2014-01,1
37,2015-02,1
49,2016-02,1
60,2017-01,1
73,2018-02,1
85,2019-02,1
96,2020-01,1
109,2021-02,1


In [10]:
dataset["MonthYear"].duplicated().sum()

np.int64(0)

## Bổ sung biến giả Covid-19

Biến `Dummy_Covid` được sử dụng để phản ánh giai đoạn dịch Covid-19 ảnh hưởng đến hoạt động đi lại và CPI nhóm Giao thông.

- `Dummy_Covid = 1`: từ tháng 01/2020 đến tháng 10/2021.
- `Dummy_Covid = 0`: các tháng còn lại.

In [11]:
dataset["Dummy_Covid"] = (
    (dataset["MonthYear"] >= pd.Period("2020-01", freq="M")) &
    (dataset["MonthYear"] <= pd.Period("2021-10", freq="M"))
).astype(int)

dataset.loc[
    dataset["Dummy_Covid"] == 1,
    ["MonthYear", "Dummy_Covid"]
]

,MonthYear,Dummy_Covid
96,2020-01,1
97,2020-02,1
98,2020-03,1
99,2020-04,1
100,2020-05,1
101,2020-06,1
102,2020-07,1
103,2020-08,1
104,2020-09,1
105,2020-10,1


In [12]:
dataset = dataset[
    [
        "MonthYear",
        "CPI",
        "RON95",
        "Diesel",
        "Brent",
        "WTI",
        "USD_VND",
        "Dummy_Tet",
        "Dummy_Covid"
    ]
]

dataset.head()

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid
0,2012-01,0.66,21800.00,21100.00,110.69,100.27,20828.0,1,0
1,2012-02,0.23,21800.00,21100.00,119.33,102.20,20828.0,0,0
2,2012-03,1.08,23090.32,21341.94,125.45,106.16,20828.0,0,0
3,2012-04,2.67,23400.00,21400.00,119.75,103.32,20828.0,0,0
4,2012-05,1.32,23522.58,21432.26,110.34,94.66,20828.0,0,0


In [13]:
dataset.to_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/processed/model_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)